# 04 - Improvement: a more robust language model underneath TMD

**Idea.** TMD is structure-free: it only needs embeddings, so it can sit on top of *any* fine-tuned model. The paper applies it to a plainly fine-tuned BERT. Here the BERT is fine-tuned on a mix of clean text and text with random counter-fitted synonym substitutions (the SAFER-style perturbation set shipped in `TextDefender/counterfitted_neighbors.json`). The encoder then maps near-synonym variants to nearby embeddings by itself, so the projection has less to repair, and the InfoGAN is trained on that model's embeddings.

**What changes vs the paper:** the language model's fine-tuning (50% of training texts get substitutions at rate rho, the attack's own modification ratio). InfoGAN architecture, hyperparameters, the projection and the attacks are unchanged.

**Fairness note (important):** this is a stronger *defense stack*, not a like-for-like improvement of the manifold idea. The paper's baselines include augmentation-style defenses; a model trained on synonym substitutions will be more robust to PWWS and TextFooler, which use the same counter-fitted neighbours, and may be less so to BERT-Attack. The notebook therefore also attacks the augmented model *without* TMD, so you can see how much of any gain comes from the augmentation alone and how much TMD adds on top.

Run notebook 00 first (for the baseline results).

In [ ]:
!pip -q install -U transformers datasets accelerate textattack tensorflow_hub pandas
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import torch, tmd_lab as T
T.setup_env()
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'   # must match notebook 00
N_EVAL = 1000
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run'
AUG_PROB = 0.5        # fraction of training texts that get substitutions

lab = T.Lab(DATASET, work=WORK)
aug = T.SynonymAugmenter(f'{DIR}/TextDefender/counterfitted_neighbors.json', rate=lab.cfg['rho'], seed=0)

# Robust encoder: its own fine-tuned model, embeddings and InfoGAN, cached under the tag 'aug'.
lab.finetune('aug', augment_fn=aug, aug_prob=AUG_PROB)
E_tr, Y_tr = lab.embeddings('train', 'aug')
E_va, Y_va = lab.embeddings('val', 'aug')
E_te, Y_te = lab.embeddings('test', 'aug')
gan = lab.get_gan('aug', E_tr, E_va)

In [ ]:
# (a) augmented model alone, no projection: isolates what the augmentation buys.
aug_only = T.Victim(lab, mode='none')
T.evaluate(lab, aug_only, 'aug_vanilla', E_te, Y_te, n=N_EVAL)

# (b) augmented model + TMD: the candidate to compare against the paper's TMD.
aug_tmd = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)
T.evaluate(lab, aug_tmd, 'aug_tmd', E_te, Y_te, n=N_EVAL)

print('\n== augmented model + TMD vs the paper and vs our baseline (plain BERT + TMD)')
display(T.compare(lab, 'aug_tmd', baseline_tag='baseline'))
print('\n== augmented model alone vs the paper')
display(T.compare(lab, 'aug_vanilla', baseline_tag='vanilla'))